# Floating Head Oil Tank Detection - Training Pipeline
YOLOv8 (Ultralytics) trained on the `towardsentropy/oil-storage-tanks` Kaggle dataset.

**Dataset to pick:** `towardsentropy/oil-storage-tanks`
(single class: floating head tank — matches the shadow-volume estimation algorithm downstream, since only floating-head tanks have the interior/exterior shadow crescent needed for volume calculation)

**Pipeline stages:**
1. Setup + Kaggle auth
2. Download dataset
3. Parse labels.json, drop 'skip' images
4. Convert to YOLO format + train/val split
5. Train YOLOv8 with Google Drive checkpointing (resume-safe across Colab disconnects)
6. Evaluate (AP / mAP)
7. Run inference + visualize detections


## Step 0: Confirm GPU runtime
Runtime > Change runtime type > GPU (T4 free tier is fine).

In [ ]:
!nvidia-smi

## Step 1: Install dependencies

In [ ]:
!pip install ultralytics kaggle -q
import ultralytics
ultralytics.checks()

## Step 2: Mount Google Drive
This is used for **checkpoint persistence** across Colab disconnects (same pattern as your other Colab-crash-recovery projects). Training will save checkpoints here every few epochs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/oil_tank_project/runs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)

## Step 3: Kaggle API authentication
1. Go to https://www.kaggle.com/settings/account
2. Click "Create New API Token" — downloads `kaggle.json`
3. Upload it below when prompted

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## Step 4: Download the dataset
**Dataset:** `towardsentropy/oil-storage-tanks`
Contains `large_images/` (100 x 4800x4800), `image_patches/` (512x512 tiles), `labels.json`, `labels_coco.json`, `large_image_data.csv`.

In [ ]:
!kaggle datasets download -d towardsentropy/oil-storage-tanks -p /content/data --unzip
!ls /content/data

## Step 5: Parse labels.json and filter out 'skip' images
Images labeled `'skip'` contain no floating-head tanks and are dropped from the training set — but keep a note of the total counts so you know your effective dataset size.

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path('/content/data')
with open(DATA_DIR / 'labels.json') as f:
    labels = json.load(f)

print("Total labeled entries:", len(labels))

# Inspect structure of the first non-skip entry
for entry in labels:
    if entry.get('label') != 'skip' and entry.get('annotations'):
        print(json.dumps(entry, indent=2)[:800])
        break

In [ ]:
# NOTE: The exact key names in labels.json can vary slightly between dataset
# versions. Run the cell above first and adjust the key names below
# (e.g. 'annotations', 'bbox', 'file_name') to match what you actually see.

usable = [e for e in labels if e.get('label') != 'skip' and e.get('annotations')]
skipped = [e for e in labels if e.get('label') == 'skip']

print(f"Usable images (contain floating-head tanks): {len(usable)}")
print(f"Skipped images (no tanks): {len(skipped)}")

## Step 6: Convert bounding boxes to YOLO format + train/val split
YOLO format per image: one `.txt` file with lines `class x_center y_center width height` (all normalized 0-1).

The dataset stores boxes as 4-corner (x,y) pairs — convert these to `[x_min, y_min, x_max, y_max]` first, then to YOLO's normalized center/width/height format.

In [ ]:
import shutil
import random
from PIL import Image

random.seed(42)
random.shuffle(usable)

split_idx = int(0.85 * len(usable))
train_entries = usable[:split_idx]
val_entries = usable[split_idx:]

print(f"Train: {len(train_entries)} | Val: {len(val_entries)}")

YOLO_ROOT = Path('/content/yolo_dataset')
for split in ['train', 'val']:
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

PATCH_DIR = DATA_DIR / 'image_patches'

def corners_to_yolo_line(corners, img_w, img_h, class_id=0):
    xs = [p[0] for p in corners]
    ys = [p[1] for p in corners]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = ((x_min + x_max) / 2) / img_w
    y_center = ((y_min + y_max) / 2) / img_h
    w = (x_max - x_min) / img_w
    h = (y_max - y_min) / img_h
    return f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

def process_split(entries, split_name):
    written = 0
    for entry in entries:
        fname = entry.get('file_name') or entry.get('image') or entry.get('id')
        if fname is None:
            continue
        if not str(fname).endswith('.jpg'):
            fname = f"{fname}.jpg"
        src_img = PATCH_DIR / fname
        if not src_img.exists():
            continue

        with Image.open(src_img) as im:
            img_w, img_h = im.size

        lines = []
        for ann in entry.get('annotations', []):
            corners = ann if isinstance(ann, list) else ann.get('bbox', ann)
            if len(corners) == 4 and isinstance(corners[0], (int, float)):
                # already [x_min, y_min, x_max, y_max]
                x_min, y_min, x_max, y_max = corners
                x_center = ((x_min + x_max) / 2) / img_w
                y_center = ((y_min + y_max) / 2) / img_h
                w = (x_max - x_min) / img_w
                h = (y_max - y_min) / img_h
                lines.append(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
            else:
                lines.append(corners_to_yolo_line(corners, img_w, img_h))

        if not lines:
            continue

        shutil.copy(src_img, YOLO_ROOT / 'images' / split_name / fname)
        label_path = YOLO_ROOT / 'labels' / split_name / (Path(fname).stem + '.txt')
        label_path.write_text("\n".join(lines))
        written += 1
    return written

n_train = process_split(train_entries, 'train')
n_val = process_split(val_entries, 'val')
print(f"Written -> train: {n_train}, val: {n_val}")

**Important:** run the Step 5 inspection cell first and check the real key names in `labels.json`
(they may be `'annotations'`, `'boxes'`, `'polygon'`, etc. depending on dataset version) and adjust
`fname = entry.get(...)` / `entry.get('annotations', [])` above to match before running the conversion.

## Step 7: Create data.yaml for YOLOv8

In [ ]:
yaml_content = f"""
path: {YOLO_ROOT}
train: images/train
val: images/val

names:
  0: floating_head_tank
"""

with open(YOLO_ROOT / 'data.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

## Step 8: Train YOLOv8 (with Drive checkpointing + resume support)
- `project=CHECKPOINT_DIR` saves weights directly to Drive, so a Colab disconnect doesn't lose progress.
- To resume after a disconnect, re-run Steps 1-2 (skip re-downloading data if patches are still on Drive), then run the **resume cell** below instead of starting fresh.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano - fastest for free-tier T4; use yolov8s.pt for better accuracy if time allows

results = model.train(
    data=str(YOLO_ROOT / 'data.yaml'),
    epochs=100,
    imgsz=512,
    batch=16,
    project=CHECKPOINT_DIR,
    name='floating_head_yolov8',
    patience=20,          # early stopping if no improvement
    save_period=5,        # checkpoint every 5 epochs
    exist_ok=True,
    device=0,
)

### Resume cell (only run this if training was interrupted)
Ultralytics auto-detects the last checkpoint in the run folder.

In [ ]:
# Only run this cell if you need to resume an interrupted run
resume_path = f"{CHECKPOINT_DIR}/floating_head_yolov8/weights/last.pt"
model = YOLO(resume_path)
results = model.train(resume=True)

## Step 9: Evaluate — AP / mAP on validation set

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## Step 10: Run inference on a few validation images and visualize

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import random

val_images = list((YOLO_ROOT / 'images' / 'val').glob('*.jpg'))
sample = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), sample):
    result = model.predict(str(img_path), conf=0.25, verbose=False)[0]
    plotted = result.plot()
    ax.imshow(plotted[..., ::-1])  # BGR -> RGB
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 11: Export final weights for use in your app
Copy the best weights to a stable location so the Streamlit app can load them directly.

In [ ]:
best_weights = f"{CHECKPOINT_DIR}/floating_head_yolov8/weights/best.pt"
!cp {best_weights} /content/drive/MyDrive/oil_tank_project/best_floating_head_yolov8.pt
print("Saved final weights to Drive:", "/content/drive/MyDrive/oil_tank_project/best_floating_head_yolov8.pt")